In [1]:
import gymnasium as gym
import random

random.seed(1234)

streets = gym.make("Taxi-v3", render_mode="ansi").env
streets.reset()
streets.render()

'+---------+\n|\x1bR\x1b: | : :G|\n| : | : : |\n| :\x1b \x1b: : : |\n| | : | : |\n|Y| : |\x1bB\x1b: |\n+---------+\n\n'

In [2]:
print(type(streets))

<class 'gymnasium.wrappers.common.OrderEnforcing'>


In [3]:
initial_state = streets.unwrapped.encode(2, 3, 2, 0)
streets.s = initial_state

streets.render()

'+---------+\n|\x1bR\x1b: | : :G|\n| : | : : |\n| :\x1b \x1b: : : |\n| | : | : |\n|Y| : |\x1bB\x1b: |\n+---------+\n\n'

In [4]:
streets.unwrapped.P[initial_state]

{0: [(1.0, 368, -1, False)],
 1: [(1.0, 168, -1, False)],
 2: [(1.0, 288, -1, False)],
 3: [(1.0, 248, -1, False)],
 4: [(1.0, 268, -10, False)],
 5: [(1.0, 268, -10, False)]}

In [5]:
import numpy as np

q_table = np.zeros([streets.observation_space.n, streets.action_space.n])

# How much new info overrides the old info
learning_rate = 0.1

# Does Something regards to immediate vs future rewards
discount_factor = 0.6

# Try something new, see the if condition
exploration = 0.1

#Repitions, i knew this one
epochs = 1000

for taxi_run in range (epochs):
    state = streets.reset()[0]
    done = False

    while not done:
        random_value = random.uniform(0,1)
        if(random_value < exploration):
            action = streets.action_space.sample() # Exploring Random actions
        else:
            action = np.argmax(q_table[state]) # Use the action with highest q-value

        # Unpack 5 Values
        next_state, reward, terminated, truncated, info = streets.step(action)

        # Combine the terminated and truncated as done
        done = truncated or terminated

        # for each row q table stores current and next steps i suppose
        prev_q = q_table[state, action]
        next_max_q = np.max(q_table[next_state])

        # Bellman Equation
        new_q = ((1-learning_rate) * prev_q) + (learning_rate * (reward + discount_factor * next_max_q))
        q_table[state, action] = new_q

        state = next_state

In [6]:
q_table[initial_state]

array([-2.38872491, -2.38981183, -2.3892629 , -2.38607511, -6.72804823,
       -5.17073585])

In [7]:
from IPython.display import clear_output
from time import sleep

for tripnum in range(1, 11):
    state = streets.reset()[0]

    done = False
    trip_length = 0

    # < 25 because no trip can cover and repeat more than one position
    while not done and trip_length < 25:
        action = np.argmax(q_table[state])
        next_state, reward, terminated, truncated, info = streets.step(action)
        done = terminated or truncated
        clear_output(wait=True)

        print("Trip number " + str(tripnum) + " Step " + str(trip_length))
        print(streets.render())
        sleep(.5)
        state = next_state
        trip_length += 1

    sleep(2)

Trip number 5 Step 24
+---------+
|R: | : :G|
| : | : : |
| : : : : |
| | : | : |
|Y| : |B: |
+---------+
  (South)



KeyboardInterrupt: 